In [1]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import random
from typing import Literal, get_args

import h5py
from hist import Hist
import matplotlib.pyplot as plt
import pandas as pd
import numba as nb
import numpy as np
from tqdm import tqdm

from h5flow.data import H5FlowDataManager

- MIP track parallel to anode and 3cm away (i.e. short current pulse on pixel). Horizontal and slicing through the pixel's center.
- MIP track highly-inclined (85 degrees) to anode, 3-6cm away (i.e. long current pulse on pixel). Horizontal and symmetric about the pixel's center.
- Two anode-parallel MIP tracks on same pixel, closely spaced in time, with separation of 2, 5, and 10 usec (closer track 3cm from anode; horizontal and slicing through the pixel's center)
- MIP track clipping pixel (i.e. small pulse close to ~5000 ke- design self-trigger threshold): Parallel to anode, 45-degrees rotated in the "pixel" plane, 3cm away, shifted upward by 3/4 of a pitch.
- HIP track on pixel (i.e. pulse with higher peak current on pixel). Same geometry as first case (parallel MIP track).
- Shower on pixel (i.e. long pulse with lots of structure)

In [2]:
import library.edep_synth as ES

In [3]:
def A(*a, **kw):
    return np.array(a, **kw)

In [4]:
os.chdir('/pscratch/sd/m/mkramer/test/larnd-sim.segFFE3')

In [5]:
!mkdir -p input/chg_inject output/chg_inject

In [6]:
mod_center = A(0, 0, 0) # fsd.yaml

In [7]:
# an important number, but not used here:
drift_len = 46.788 # multi_tile_layout-3.0.40_fsd_v3.yaml

In [8]:
# instead we use this:
tile_x = 47.1055 # multi_tile_layout-3.0.40_fsd_v3.yaml

In [9]:
# sanity check: this should be the half-width of the cathode
tile_x - drift_len

0.31750000000000256

In [21]:
pitch = 0.372 # multi_tile_layout-3.0.40_fsd_v3.yaml

In [22]:
dest = 'input/chg_inject'

### Scenarios

In [23]:
def gen_parallel_mip():
    "MIP track parallel to anode and 3cm away (i.e. short current pulse on pixel). Horizontal and slicing through the pixel's center."
    mid = mod_center + A(tile_x - 3, pitch/2, 0)
    start = mid - A(0, 0, 3*pitch)
    end = mid + A(0, 0, 3*pitch)

    ES.write_segments(f'{dest}/parallel_mip.hdf5', [(start, end)], dEdx=2)

gen_parallel_mip()

In [24]:
!ls -l $dest

total 1
-rw-rw-r-- 1 mkramer dune 6744 Aug  6 18:37 parallel_mip.hdf5
